## Clustering


Type of glass: (class attribute)
- 1 building_windows_float_processed
- 2 building_windows_non_float_processed
- 3 vehicle_windows_float_processed
- 4 vehicle_windows_non_float_processed (none in this database)
- 5 containers
- 6 tableware
- 7 headlamps
- For more information on the dataset, refer https://archive.ics.uci.edu/dataset/42/glass+identification

- RI	refractive index
- Na	Sodium
- Mg	Magnesium	
- Al	Aluminum
- Si	Silicon	
- K	Potassium	
- Ca	Calcium	
- Ba	Barium	
- Fe	Iron	
- Type_of_glass	Target	Categorical	

In [ ]:
# importing packages

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans

from scipy.cluster.hierarchy import linkage , dendrogram, cut_tree

from sklearn.preprocessing import StandardScaler

from sklearn.neighbors import NearestNeighbors
from random import sample
from numpy.random import uniform
import numpy as np
from math import isnan
from sklearn.metrics import silhouette_score
from sklearn.metrics import accuracy_score


%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('glass.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
## Drop Type column
df.drop('Type', axis = 1, inplace = True)
df.head()

In [ ]:
df['ID'] = 100+df.index
df.head()

In [ ]:
# Check if any null values exist in the dataset
df.isnull().sum()

In [ ]:
## Outlier Treatment
col = df.columns
col

In [ ]:
plt.figure(figsize = (15,30))
col = ['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe']
for idx, i in enumerate(col, start=1):
    plt.subplot(5,2,idx)
    sns.boxplot(x = i, data = df)

In [ ]:
for i in col:
    q1 = df[i].quantile(0.01)
    q4 = df[i].quantile(0.99)
    df = df[(df[i]>=q1) & (df[i]<=q4)]

# As part of the hands-on task, try to implement Tukey fences (k=1.5) similar to the above methodlogy

In [ ]:
plt.figure(figsize = (15,30))
col = ['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe']
for i in enumerate(col):
    plt.subplot(5,2,i[0]+1)
    sns.boxplot(x = i[1], data = df)

In [ ]:
df.shape

In [ ]:
## Scale
df.head()

In [ ]:
dat1 = df.drop('ID', axis=1)
dat1.head()

In [ ]:
scaler = StandardScaler()
dat2  =scaler.fit_transform(dat1)

In [ ]:
# Check the type of dat2
type(dat2)

In [ ]:
dat2 = pd.DataFrame(dat2)
dat2.columns = dat1.columns
dat2.head()

In [ ]:
dat2.describe()

In [ ]:
sns.pairplot(dat2)

## Tendency of Clustering

In [ ]:
uniform(np.amin(dat2,axis=0),np.amax(dat2,axis=0),dat2.shape[1]).reshape(1, -1)

In [ ]:
## Clustering
# Check whether the given dataset is clusterable

def hopkins(X):
    d = X.shape[1] # columns
    n = len(X) # rows
    m = int(0.1 * n) 
    nbrs = NearestNeighbors(n_neighbors=1).fit(X.values)
 
    rand_X = sample(range(0, n, 1), m)
 
    ujd = []
    wjd = []
    for j in range(0, m):
        #Random data
        u_dist, _ = nbrs.kneighbors(uniform(np.amin(X,axis=0),np.amax(X,axis=0),d).reshape(1, -1), 2, return_distance=True)
        ujd.append(u_dist[0][1])
        
        #From the dataset
        w_dist, _ = nbrs.kneighbors(X.iloc[rand_X[j]].values.reshape(1, -1), 2, return_distance=True)
        wjd.append(w_dist[0][1])
 
    H = sum(ujd) / (sum(ujd) + sum(wjd))
    if isnan(H):
        print(ujd, wjd)
        H = 0
 
    return H

In [ ]:
hopkins(dat2)

In [ ]:
data_main=dat2

## K-Mean

### Cluster= 3

In [ ]:
kmean3 = KMeans(n_clusters = 3, random_state = 50).fit(dat2)

In [ ]:
kmean3.labels_

In [ ]:
# Cluster the data
sil = []
for k in range(2,10):
    kmean = KMeans(n_clusters = k, random_state=42).fit(dat2)
    sil.append([k, silhouette_score(dat2, kmean.labels_)])

In [ ]:
plt.plot(pd.DataFrame(sil)[0], pd.DataFrame(sil)[1])

In [ ]:
ssd = []
for k in range(2, 10):
    k_mean =KMeans(n_clusters = k, random_state=50).fit(dat2)
    ssd.append([k, k_mean.inertia_])
    
plt.plot(pd.DataFrame(ssd)[0], pd.DataFrame(ssd)[1])

In [ ]:
dat2['ID'] = df['ID'].reset_index().drop('index',axis=1)
dat2.head()

In [ ]:
dat_km_scale = pd.concat([dat2, pd.Series(kmean3.labels_)], axis=1)
dat_km_scale.head()
dat_km_scale.columns = ['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'ID', 'Cluster_id']

In [ ]:
dat_km_scale.head()

In [ ]:
dat_km_scale.Cluster_id.value_counts()

In [ ]:
sns.scatterplot(x = 'RI', y  = 'Na', data = dat_km_scale, hue = 'Cluster_id', palette = ['green', 'orange', 'brown'])

In [ ]:
sns.scatterplot(x = 'RI', y  = 'Ba', data = dat_km_scale, hue = 'Cluster_id', palette = ['green', 'orange', 'brown'])

In [ ]:
## Cluster Profiling
dat_km = pd.merge(df, dat_km_scale[['ID', 'Cluster_id']], on = 'ID')
dat_km.head()

In [ ]:
kpi_subset = dat_km[['K', 'Ca', 'Ba', 'Fe', 'Cluster_id']]
kpi_subset.head()

In [ ]:
plt.figure(figsize = (15, 10))
features = ['K', 'Ca', 'Ba', 'Fe']
for i in enumerate(features):
    plt.subplot(2,2,i[0]+1)
    sns.boxplot(x ='Cluster_id', y= i[1], data = kpi_subset)

In [ ]:
df_test = pd.read_csv('glass.csv')
df_test['ID'] = 100+df_test.index
df_test.head()

In [ ]:
test = pd.merge(dat_km, df_test[['ID', 'Type']], on = 'ID')
test.head()

In [ ]:
test[test['Cluster_id']==0]['Type'].value_counts().plot(kind = 'bar')

In [ ]:
test[test['Cluster_id']==1]['Type'].value_counts().plot(kind = 'bar')

In [ ]:
test[test['Cluster_id']==2]['Type'].value_counts().plot(kind = 'bar')

## Clusters = 6 based on the elbow plot

In [ ]:
kmean6 = KMeans(n_clusters = 6, random_state = 50).fit(dat2)

In [ ]:
kmean6.labels_

In [ ]:
dat2 = data_main ## Going back to original

In [ ]:
dat2['ID'] = df['ID'].reset_index().drop('index',axis=1)
dat2.head()

In [ ]:
dat_km_scale = pd.concat([dat2, pd.Series(kmean6.labels_)], axis =1)
dat_km_scale.head()
dat_km_scale.columns = ['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'ID', 'Cluster_id']

In [ ]:
dat_km_scale.head()

In [ ]:
sns.scatterplot(x = 'RI', y  = 'Na', data = dat_km_scale, hue = 'Cluster_id', palette = ['green', 'orange', 'brown','blue','yellow'])

In [ ]:
dat_km_scale.Cluster_id.value_counts()

In [ ]:
sns.scatterplot(x = 'RI', y  = 'Ba', data = dat_km_scale, hue = 'Cluster_id', palette = ['green','orange','brown','red','blue'])

In [ ]:
## Cluster Profiling
dat_km = pd.merge(df, dat_km_scale[['ID', 'Cluster_id']], on = 'ID')
dat_km.head()

In [ ]:
kpi_subset = dat_km[['K', 'Ca', 'Ba', 'Fe', 'Cluster_id']]
kpi_subset.head()

In [ ]:
plt.figure(figsize = (15, 10))
features = ['K', 'Ca', 'Ba', 'Fe']
for i in enumerate(features):
    plt.subplot(2,2,i[0]+1)
    sns.boxplot(x ='Cluster_id', y= i[1], data = kpi_subset)

In [ ]:
df_test = pd.read_csv('glass.csv')
df_test['ID'] = 100+df_test.index
df_test.head()

In [ ]:
test = pd.merge(dat_km, df_test[['ID', 'Type']], on = 'ID')
test.head(5)

In [ ]:
test[test['Cluster_id']==0]['Type'].value_counts().plot(kind = 'bar')

In [ ]:
test[test['Cluster_id']==1]['Type'].value_counts().plot(kind = 'bar')

In [ ]:
test[test['Cluster_id']==2]['Type'].value_counts().plot(kind = 'bar')

In [ ]:
test[test['Cluster_id']==3]['Type'].value_counts().plot(kind = 'bar')

In [ ]:
test[test['Cluster_id']==4]['Type'].value_counts().plot(kind = 'bar')

In [ ]:
test[test['Cluster_id']==5]['Type'].value_counts().plot(kind = 'bar')

### ######End of KMeans algorithm##########

## Heirarchical Clustering
### Agglomerative (bottom-up) hierarchical clustering

In [ ]:
## Heirarchical Clustering
plt.figure(figsize=(25,10))
# method = 'single', distance between closest pair of points in clusters
mergings = linkage(dat2.drop('ID', axis  =1), method = 'single', metric = 'euclidean')
dendrogram(mergings)
plt.show()

In [ ]:
# method = 'complete', distance between farthest pair
plt.figure(figsize=(25,10))
mergings = linkage(dat2.drop('ID', axis  =1), method = 'complete', metric = 'euclidean')
dendrogram(mergings)
plt.show()

In [ ]:
# method = 'average', average distance between all pairs of data points
plt.figure(figsize=(25,10))
mergings = linkage(dat2.drop('ID', axis  =1), method = 'average', metric = 'euclidean')
dendrogram(mergings)
plt.show()

In [ ]:
# method = 'ward', minimizes the variance of the clusters being merged.
plt.figure(figsize=(25,10))
mergings = linkage(dat2.drop('ID', axis  =1), method = 'ward', metric = 'euclidean')
dendrogram(mergings)
plt.show()

## Clusters = 6 based on the elbow plot

In [ ]:
h_label = pd.Series(cut_tree(mergings, n_clusters =6).reshape(-1,))

In [ ]:
h_label

In [ ]:
dat2 = data_main
dat2.columns

In [ ]:
dat_h = pd.concat([dat2, h_label], axis = 1)
dat_h.columns = ['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'ID', 'Cluster_id']
dat_h.head()

In [ ]:
dat_h.Cluster_id.value_counts()

In [ ]:
test = pd.merge(dat_h, df_test[['ID', 'Type']], on = 'ID')
test.head()

In [ ]:
test[test['Cluster_id']==0]['Type'].value_counts().plot(kind = 'bar')

In [ ]:
test[test['Cluster_id']==1]['Type'].value_counts().plot(kind = 'bar')

In [ ]:
test[test['Cluster_id']==2]['Type'].value_counts().plot(kind = 'bar')

In [ ]:
# Try out the above analysis for different values of k other than 6

#### Neither scipy nor scikit-learn provides a direct, built-in implementation of the divisive clustering